# MLC26: multi-pipeline blocker + LightGBM matcher on Kaggle

Runs `scripts/run_multiblock.sh` (see `docs/BLOCKING.md`). Edit **Settings**, then *Run All* or *Save Version* (headless, up to 12 h).

**One-time prep**
1. **Data:** upload the official `student_resource` zip as a *private* Kaggle Dataset (Kaggle extracts it). Alternatively, upload `mlc26_data.tar.zst` from `scripts/pack_data.sh`. Attach it to this notebook with *Add Input*.
2. **Repository:** it is cloned from `REPO_URL`. No token is needed while the repo is public. For a private repo, add a read-only token as the secret `GH_TOKEN` (*Add-ons → Secrets*).
3. **Settings → Internet: On.** It is needed for `pip` and the Hugging Face model downloads.
4. **Accelerator:**
   - **None/CPU** for `MODE='classical'`;
   - **GPU T4 x2** for `MODE='full'`. Both GPUs are used automatically.

**Recommended plan**

| Run | Accelerator | Settings | Gives you |
|---|---|---|---|
| A | CPU | `MODE='classical'` | Safe first submission, no neural training |
| B | GPU T4 x2 | `MODE='full'`, `STAGES='splits train'` | Trained encoders in `/kaggle/working/artifacts` |
| C | GPU T4 x2 | `MODE='full'`, `STAGES='splits block_val check_val fit block_test predict validate'`, `PREV_ARTIFACTS=<run B output>` | Full submission |

Run B+C together as one run with `STAGES=''` if you trust it to finish in 12 h. The estimate is ~6–8 h, but nothing has been timed yet.

For run C, add run B's saved output as an input (*Add Input → Your Work → the notebook's version*), then set `PREV_ARTIFACTS` to its `artifacts` folder.

In [ ]:
# ---- Settings ---------------------------------------------------------------
REPO_URL = 'https://github.com/rishang19dx/AMZN_MLC2026.git'   # public repo (or a fork)
BRANCH = 'karanpreet'
MODE = 'classical'          # 'classical' (CPU, no neural nets) | 'full' (classical + BERT + JEPA, needs GPU)
STAGES = ''                 # '' = all; any of: splits train block_val check_val fit block_test predict validate
TRAIN_FRACTION = 0.2        # share of local_train Source 1 entities used to fine-tune the encoders
EPOCHS = 2                  # epochs per encoder
MATCHER_FRACTION = 1.0      # share of local_val entities the LightGBM matcher trains on
PREDICT_FOLDS = 0           # fold models averaged on test (0 = all 5, as submitted); 1-2 = faster, approximate
PREV_ARTIFACTS = None       # e.g. '/kaggle/input/<previous-run>/artifacts' to reuse trained encoders
RUN_TESTS = True            # tests/test_blocker.py on synthetic data (~1-2 min) before the real run
OVERRIDES = {               # merged over configs/blocking.yaml, e.g. {'pipelines': {'bert': {'top_k_source2': 30}}}
}

In [ ]:
import os, glob, json, shutil, subprocess
assert 'KAGGLE_KERNEL_RUN_TYPE' in os.environ, 'this notebook is for Kaggle'
REPO = '/tmp/MLC26'
PROJ = f'{REPO}/code/business_entity_resolution'
DATA = '/tmp/data'                      # splits are written here (inputs are read-only)
OUT = '/tmp/out'                        # big intermediates (debug parquets, embeddings)
WORK = '/kaggle/working'
ART = f'{WORK}/artifacts'               # trained encoders, translit dictionary, matcher: kept by Save Version
FINAL = f'{WORK}/output'                # submission files + reports

archives = glob.glob('/kaggle/input/**/mlc26_data.tar.zst', recursive=True)
raw = glob.glob('/kaggle/input/**/train/train_source1.tsv', recursive=True)
if archives:
    DATA_SRC = archives[0]
elif raw:
    DATA_SRC = os.path.dirname(os.path.dirname(raw[0]))   # the folder holding train/ and test/
else:
    raise SystemExit('attach the challenge data (student_resource zip or mlc26_data.tar.zst) as an input')
print('data source:', DATA_SRC, '| mode:', MODE, '| stages:', STAGES or 'all')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'no GPU'
!nproc; free -g | head -2; df -h /tmp /kaggle/working | tail -2
if MODE == 'full':
    import torch
    assert torch.cuda.is_available(), "MODE='full' needs a GPU accelerator (GPU T4 x2)"

In [ ]:
# public repo: plain clone. Private repo: add a Kaggle secret GH_TOKEN (read access) and it is used.
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    pass
if not os.path.exists(REPO):
    url = REPO_URL.replace('https://', f'https://{token}@') if token else REPO_URL
    r = subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], env=dict(os.environ, GIT_TERMINAL_PROMPT='0'))
    if r.returncode:
        raise SystemExit(f'clone failed: is {REPO_URL} public (open it in a private browser window), '
                         'or add a GH_TOKEN secret? Is Internet on?')
    subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', REPO_URL], check=True)   # no token in .git/config
!git -C {REPO} log --oneline -1


In [ ]:
# data -> /tmp/data, pinned requirements (keeps Kaggle's CUDA torch), local_train / local_val splits
os.environ['BER_DATA_DIR'] = DATA
os.environ['BER_CACHE_DIR'] = '/tmp/cache'
os.environ['BER_OUTPUT_DIR'] = OUT
!bash {PROJ}/scripts/cloud_setup.sh {DATA_SRC}

In [ ]:
if RUN_TESTS:
    !cd {PROJ} && python -m pytest tests/test_blocker.py -q -x

In [ ]:
os.makedirs(ART, exist_ok=True)
os.makedirs(FINAL, exist_ok=True)   # deliverables are copied here after each stage
if PREV_ARTIFACTS:
    shutil.copytree(PREV_ARTIFACTS, ART, dirs_exist_ok=True)
    print('restored artifacts:', sorted(os.listdir(ART)))

import yaml
cfg_path = f'{WORK}/kaggle_blocking.yaml'
with open(cfg_path, 'w') as f:
    yaml.safe_dump(OVERRIDES or {}, f)

env = dict(os.environ, DATA=DATA, ART=ART, OUT=OUT, CONFIG=cfg_path, PY='python',
           PIPELINES='classical' if MODE == 'classical' else 'classical,bert,jepa',
           TRAIN_FRACTION=str(TRAIN_FRACTION), EPOCHS=str(EPOCHS), MATCHER_FRACTION=str(MATCHER_FRACTION),
           STAGES=STAGES, SAVE_DIR=FINAL, PREDICT_FOLDS=str(PREDICT_FOLDS))
p = subprocess.Popen(['bash', f'{PROJ}/scripts/run_multiblock.sh'], env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1)
with open(f'{WORK}/run.log', 'a') as log:
    for line in p.stdout:
        print(line, end='')
        log.write(line)
assert p.wait() == 0, 'run_multiblock.sh failed; see the log above / run.log'

In [ ]:
# collect deliverables into /kaggle/working/output (kept by Save Version)
os.makedirs(FINAL, exist_ok=True)
for src in [f'{OUT}/test/matching_results.tsv', f'{OUT}/test/candidate_pairs.tsv',
            f'{OUT}/test/candidate_manifest.json', f'{OUT}/local_val/blocking_report.txt',
            f'{OUT}/local_val/blocking_report.json', f'{OUT}/local_val/check_blocking.json',
            f'{OUT}/local_val/matching_results_oof.tsv']:
    if os.path.exists(src):
        dst = f"{FINAL}/{'local_val_' if '/local_val/' in src else ''}{os.path.basename(src)}"
        shutil.copy(src, dst)
        print(f'{dst}  ({os.path.getsize(dst) / 1e6:.1f} MB)')
if os.path.exists(f'{OUT}/local_val/blocking_report.txt'):
    print(open(f'{OUT}/local_val/blocking_report.txt').read())
print('artifacts:', sorted(os.listdir(ART)) if os.path.exists(ART) else None)